<a href="https://colab.research.google.com/github/rhodes-byu/cs-stat-180/blob/main/notebooks/18-ai-nba-stats-copilot-github-models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Basketball NBA Stats Chatbot
### GitHub Models Lab (practical GitHub/Copilot alternative)

**Pattern:** question -> model calls a local pandas tool -> we run it -> model explains results

**Dataset:** [NBA Players Stats Since 1950](https://www.kaggle.com/datasets/drgilermo/nba-players-stats)

**Access:** [GitHub Models](https://github.com/marketplace/models)

**Important:** GitHub Copilot itself is not a clean, general-purpose Python notebook API for this workflow. The most practical GitHub-hosted replacement is GitHub Models, which uses a PAT with `models` scope and includes free, rate-limited usage for experimentation.

## Load in the data from Kaggle

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("drgilermo/nba-players-stats")

print("Path to dataset files:", path)

## Install the small helper dependency if needed

In [ ]:
!pip install requests --quiet
print("Done")

In [ ]:
import pandas as pd, numpy as np, io


df = pd.read_csv(path + '/Seasons_Stats.csv')
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df = df.dropna(subset=["Year"]).copy()
df["Year"] = df["Year"].astype(int)
df["Player"] = df["Player"].astype(str).str.strip().str.replace("*", "", regex=False)
stat_cols = ["G","GS","MP","PTS","TRB","ORB","DRB","AST","STL","BLK","TOV","PF",
             "FG","FGA","FG%","3P","3PA","3P%","2P","2PA","2P%","FT","FTA","FT%",
             "eFG%","TS%","PER","USG%","WS","WS/48","OWS","DWS","BPM","VORP","Age"]
for col in stat_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Loaded {len(df):,} rows - {df['Player'].nunique():,} players, {df['Year'].min()}-{df['Year'].max()}")
df.head(3)


## We will need a GitHub personal access token

Create a PAT with the `models` scope, then paste it below.

In [ ]:
import getpass
API_KEY = getpass.getpass("Paste your GitHub PAT (models scope): ")

## The Data Dictionary

Instead of hardcoded keyword rules, we give the LLM a plain-English description
of every column so it can write the right pandas query for any question.

Getting this right matters: the original version said BLK was 'blocks per game'
when it's actually a season total â€” so the LLM reported Mark Eaton averaging
278 blocks per game. The data dictionary is the LLM's only view of the schema.


In [ ]:
# Data dictionary injected into every prompt.
# Sourced from the official Basketball Reference glossary.

lines = [
    "DataFrame: df | ~24,000 rows | one row per player per season",
    "Year: end year of season (2000 = the 1999-2000 season)",
    "Player: full name string e.g. Michael Jordan",
    "Pos: position - PG, SG, SF, PF, C (sometimes combined e.g. SF-PF)",
    "Age: player age on January 31 of the given season",
    "Tm: team abbreviation e.g. CHI, LAL, BOS. TOT = season total row for mid-season trades",
    "",
    "G: games played that season",
    "GS: games started that season (available since 1982 only)",
    "MP: total minutes played that season (NOT per game)",
    "",
    "=== COUNTING STATS: ALL ARE SEASON TOTALS, NOT PER GAME ===",
    "To get per-game rate you MUST divide by G. Example: df['BLK']/df['G'] = blocks per game",
    "",
    "PTS: total points scored that season",
    "TRB: total rebounds that season (available since 1950-51)",
    "ORB: offensive rebounds that season (available since 1973-74 only - NaN before that)",
    "DRB: defensive rebounds that season (available since 1973-74 only - NaN before that)",
    "AST: total assists that season",
    "STL: total steals that season (available since 1973-74 only - NaN before that)",
    "BLK: total blocks that season (available since 1973-74 only - NaN before that)",
    "TOV: total turnovers that season (available since 1977-78 only - NaN before that)",
    "PF: personal fouls that season",
    "FG: field goals made (both 2-point and 3-point)",
    "FGA: field goal attempts",
    "3P: three-point field goals made (available since 1979-80 only - NaN before that)",
    "3PA: three-point field goal attempts (available since 1979-80 only)",
    "2P: two-point field goals made",
    "2PA: two-point field goal attempts",
    "FT: free throws made",
    "FTA: free throw attempts",
    "",
    "=== RATE/EFFICIENCY STATS: ALREADY NORMALIZED, DO NOT DIVIDE BY G ===",
    "",
    "FG%: field goal percentage = FG/FGA",
    "3P%: three-point percentage = 3P/3PA (column name is '3P%', available since 1979-80)",
    "2P%: two-point field goal percentage = 2P/2PA",
    "FT%: free throw percentage = FT/FTA",
    "eFG%: effective field goal pct = (FG + 0.5*3P)/FGA (adjusts for 3s being worth more)",
    "TS%: true shooting pct = PTS/(2*TSA) where TSA=FGA+0.44*FTA (best overall shooting efficiency)",
    "PER: Player Efficiency Rating, per-minute rating (league average = 15, higher is better)",
    "USG%: usage percentage, estimate of % of team plays used while on floor (avg ~20%)",
    "",
    "=== ACCUMULATION STATS: SEASON VALUES, DO NOT DIVIDE BY G ===",
    "",
    "WS: Win Shares, estimate of wins contributed that season (already a season total)",
    "WS/48: Win Shares per 48 minutes (league average ~0.100, already a rate)",
    "OWS: Offensive Win Shares that season",
    "DWS: Defensive Win Shares that season",
    "BPM: Box Plus/Minus, points per 100 possessions above league average (already a rate)",
    "VORP: Value Over Replacement Player, prorated to 82-game season (already normalized, do not divide by G)",
    "",
    "=== QUERY PATTERNS ===",
    "",
    "Per-game career leaders (add column, then groupby):",
    "  tmp=df.copy(); tmp['BPG']=tmp['BLK']/tmp['G']",
    "  result=tmp.groupby('Player')['BPG'].mean().sort_values(ascending=False).head(20)",
    "",
    "Season leaderboard (exclude TOT rows to avoid double-counting traded players):",
    "  season=df[(df['Year']==YYYY)&(df['Tm']!='TOT')]",
    "  result=season.nlargest(20, 'PTS')[['Player','Tm','G','PTS']]",
    "",
    "Player career lookup (use str.contains for fuzzy match):",
    "  result=df[df['Player'].str.contains('Jordan',case=False,na=False)].sort_values('Year')",
    "",
    "Multi-player comparison:",
    "  result=df[df['Player'].str.contains('Jordan|Bryant',case=False,na=False)]",
    "",
    "Dataset covers 1950-2017. Post-2017 players are not present.",
    "Many advanced stats (BLK, STL, ORB, DRB, TOV, 3P) are NaN before their availability year.",
]

DATA_DICTIONARY = "\n".join(lines)
print(f"Data dictionary ready ({len(DATA_DICTIONARY.split())} words)")


## The `ask()` function

This version uses the GitHub Models chat completions endpoint.

Two stages:
- **Stage 1** the model returns a tool/function call request for `run_pandas_query`
- **Stage 2** we execute the tool locally, send its output back as a tool message, and ask for the final explanation

Use `debug=True` to see the generated pandas code.

In [ ]:
import json, requests, traceback

MODEL = "openai/gpt-4.1-mini"
API_URL = "https://models.github.ai/inference/chat/completions"

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "run_pandas_query",
            "description": "Run short Python/pandas code against df and return a compact preview of the result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "Executable Python that uses only df, pd, and np, assigns the final object to result, and stays within 1-6 lines."
                    }
                },
                "required": ["code"],
                "additionalProperties": False
            }
        }
    }
]

def build_prompt(question):
    return (
        "You are analyzing a pandas DataFrame named `df` with NBA player-season data.\n\n"
        + DATA_DICTIONARY + "\n\n"
        + "Use the `run_pandas_query` tool exactly once before answering.\n"
        + "The tool should receive executable Python that uses only `df`, `pd`, and `np`.\n"
        + "After the tool returns, answer the student's question using ONLY the returned data.\n\n"
        + f"Question: {question}\n\n"
        + "Rules for the tool code:\n"
        + "- Assign the final object to a variable named `result`\n"
        + "- Return only 1-6 lines of executable Python\n"
        + "- NEVER divide rate or efficiency stats (WS, VORP, PER, BPM, FG%, TS%, etc.) by G\n"
        + "- ALWAYS divide counting stats (PTS, BLK, TRB, AST, etc.) by G for per-game rates\n"
        + "- For per-game leaderboards: add a per-game column first, then groupby and mean()\n"
        + "- For season leaderboards: exclude TOT rows with df[df['Tm'] != 'TOT']\n"
        + "- For player lookup: use str.contains(case=False, na=False)\n"
        + "- For multi-player comparisons: use pipe in str.contains, e.g. 'Jordan|Bryant'\n"
        + "- If the tool result contains an error, explain the problem instead of guessing\n"
        + "- Final answer style: conversational, specific, 2-4 short paragraphs\n"
    )


def normalize_code(text):
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        text = "\n".join(lines[1:])
        if text.endswith("```"):
            text = "\n".join(text.splitlines()[:-1])
    return text.strip()


def run_pandas_query(code, debug=False):
    code = normalize_code(code)

    if debug:
        print("-- Generated code ---------------------------------")
        print(code)
        print("---------------------------------------------------\n")

    local_vars = {"df": df, "np": np, "pd": pd}

    try:
        exec(code, {}, local_vars)
        result = local_vars.get("result", None)

        if result is None:
            payload = {
                "status": "error",
                "code": code,
                "data_context": "Query produced no `result` variable.",
            }
        elif isinstance(result, pd.DataFrame):
            payload = {
                "status": "ok",
                "code": code,
                "data_context": f"Result ({len(result)} rows):\n{result.head(30).to_string(index=False)}",
            }
        elif isinstance(result, pd.Series):
            payload = {
                "status": "ok",
                "code": code,
                "data_context": f"Result:\n{result.head(30).to_string()}",
            }
        else:
            payload = {
                "status": "ok",
                "code": code,
                "data_context": f"Result: {result}",
            }
    except Exception as e:
        if debug:
            traceback.print_exc()
        payload = {
            "status": "error",
            "code": code,
            "data_context": f"Code failed ({e}). Dataset sample:\n{df.sample(15, random_state=0).to_string(index=False)}",
        }

    return payload

SYSTEM_PROMPT = (
    "You are an NBA analyst. "
    "You must use the provided function tool before answering and then answer using only the tool output."
)


def call_github_models(messages, tool_choice):
    response = requests.post(
        API_URL,
        headers={
            "Accept": "application/vnd.github+json",
            "Authorization": f"Bearer {API_KEY}",
            "X-GitHub-Api-Version": "2022-11-28",
            "Content-Type": "application/json",
        },
        json={
            "model": MODEL,
            "messages": messages,
            "tools": TOOLS,
            "tool_choice": tool_choice,
            "temperature": 0,
        },
        timeout=120,
    )
    response.raise_for_status()
    return response.json()


def ask(question, debug=False):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_prompt(question)},
    ]

    first = call_github_models(messages, tool_choice="required")
    assistant_message = first["choices"][0]["message"]
    tool_calls = assistant_message.get("tool_calls", [])

    if not tool_calls:
        print(assistant_message.get("content", "No response returned."))
        return

    messages.append(assistant_message)

    for tool_call in tool_calls:
        args = json.loads(tool_call["function"]["arguments"])
        tool_result = run_pandas_query(args["code"], debug=debug)
        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call["id"],
                "content": json.dumps(tool_result),
            }
        )

    final = call_github_models(messages, tool_choice="none")
    print(final["choices"][0]["message"]["content"])


print("Ready. Call ask('question') - add debug=True to see the generated code.")

## Ask questions

In [ ]:
ask("How did Michael Jordan perform over his career?")

In [ ]:
ask("Who were the top scorers in the 1988 season?")

In [ ]:
ask("Compare LeBron James and Kobe Bryant")

In [ ]:
ask("Which centers had the highest career blocks per game?", debug=True)

In [ ]:
ask("Who had the best true shooting percentage with at least 5 seasons?")

In [ ]:
ask("YOUR QUESTION HERE")

## Discussion

1. Run the blocks question with `debug=True`. What pandas code did the model send through the tool?
2. Why force a tool/function call before the model answers in plain English?
3. Try `ask('Who had the highest VORP?')`. Does the code correctly avoid dividing VORP by `G`?
4. What happens if the data dictionary is wrong or incomplete?
5. We still run model-written code with `exec()`. Why is that dangerous in production?